In [1]:
from pathlib import Path
from math import ceil, sqrt
from tqdm import tqdm

import numpy as np
import scipy

import pickle
import tifffile

import matplotlib.pyplot as plt
import seaborn as sns
import fastplotlib as fpl

from ipywidgets import VBox, HBox, Layout, Select, IntSlider, Dropdown

import pygpcca

To silence this warning, use a fully namespaced name.
To silence this warning, use a fully namespaced name.
To silence this warning, use a fully namespaced name.
To silence this warning, use a fully namespaced name.
To silence this warning, use a fully namespaced name.
To silence this warning, use a fully namespaced name.


Image(value=b'\x89PNG\r\n\x1a\n\x00\x00\x00\rIHDR\x00\x00\x01,\x00\x00\x007\x08\x06\x00\x00\x00\xb6\x1bw\x99\x…

Unable to find extension: VK_EXT_swapchain_colorspace


Available devices:
✅ (default) | AMD RADV POLARIS10 (ACO) | DiscreteGPU | Vulkan | Mesa 20.3.5 (ACO)
❗ | llvmpipe (LLVM 11.0.1, 256 bits) | CPU | Vulkan | Mesa 20.3.5 (LLVM 11.0.1)
✅ | NVIDIA GeForce RTX 3080 | DiscreteGPU | Vulkan | 530.30.02
❗ | Radeon RX 570 Series (POLARIS10, DRM 3.40.0, 5.10.0-21-amd64, LLVM 11.0.1) | Unknown | OpenGL | 4.6 (Core Profile) Mesa 20.3.5


In [2]:
%matplotlib inline

In [3]:
vd1 = {
    "path": Path("/home/kushal/lab/datasets/jason/M19/D05/M19D5_MC.tif"),
    "rects": [(162, 268, 178, 288), (304, 410, 118, 228), (345, 451, 297, 407)],
    "results": list()
}

vd2 = {
    "path": Path("/home/kushal/lab/datasets/jason/61/D05/61D5_MC_mmap.tif"),
    "rects": [(132, 290, 92, 264), (296, 461, 197, 414)],
    "results": list()
}

vd3 = {
    "path": Path("/home/kushal/lab/datasets/jason/885/D05/885D5_MC.tif"),
    "rects": [(155, 307, 166, 326), (231, 383, 99, 259), (126, 278, 91, 251), (47, 210, 197, 357), (179, 328, 270, 398)],
    "results": list()
}

vids = [vd1, vd2, vd3]

- view the movie subset
- percentile filtered
- line plot to click lines and show selected raw pixel and percentile filtered
- min chi line plot
- image grid of clusters, percentile filter, intslider
- linear selector on min chi to update number of clusters
- single overlaps scatter plot with random colors, use percentile filtered cluster images

In [ ]:
import numba

@numba.guvectorize([(numba.float32[:], numba.float32[:])], "(n)->(n)")
def rolling_percentile(trace: np.ndarray, out: np.ndarray):
    for i in range(0, trace.size - 1):
        out[i] = np.percentile(trace[i:i + 30], 5)


@numba.njit(parallel=True)
def rolling_percentile_movie(m, output):
    t, row, col = m.shape
    for i in numba.prange(row):
        for j in range(col):
            rolling_percentile(m[:, i, j], output[:, i, j])

In [ ]:
movie_selector = Select(
    options=[v["path"].stem for v in vids],
    description="movie: "
)

CURRENT_VID = vids[movie_selector.index]
CURRENT_RECTS = CURRENT_VID["rects"]

movie_raw = tifffile.imread(CURRENT_VID["path"])

iw_movie = fpl.ImageWidget(movie_raw, cmap="gnuplot2", figure_kwargs={"size": (582, 744)}, names=["full movie"])

for i, rect in enumerate(CURRENT_RECTS):
    rs = iw_movie.managed_graphics[0].add_rectangle_selector(rect, edge_color="magenta", edge_thickness=2, resizable=False)

rect_slider = IntSlider(value=0, min=0, max=len(CURRENT_VID["rects"]) - 1, description="rect")

BLOCK = False

def set_movie(change):
    global CURRENT_VID

    BLOCK = True
    
    index = change["new"]
    CURRENT_VID = vids[index]

    global movie_raw
    movie_raw = tifffile.imread(CURRENT_VID["path"])
    iw_movie.set_data(movie_raw)

    global CURRENT_RECTS
    CURRENT_RECTS = CURRENT_VID["rects"]
    rect_slider.value = 0
    rect_slider.max = len(CURRENT_VID["rects"] - 1)

    for s in iw_movie.figure[0, 0].selectors:
        iw_movie.figure[0, 0].delete_graphic(s)
    
    for i, rect in enumerate(CURRENT_RECTS):
        rs = iw_movie.managed_graphics[0].add_rectangle_selector(rect, edge_color="magenta", edge_thickness=2, resizable=False)
    
    n_clusters_slider.value = 0

    BLOCK = False
    
    set_rect({"new": 0})
    set_clusters({"new": 10})
    update_labels_percentile({"new": comps_percentile_slider.value})


movie_selector.observe(set_movie, "index")

CURRENT_RECT = CURRENT_RECTS[rect_slider.value]

c0, c1, r0, r1 = CURRENT_RECT

sub_raw = movie_raw[:, r0:r1:2, c0:c1:2].astype(np.float32).copy()
sub = np.zeros(sub_raw.shape, dtype=np.float32)
rolling_percentile_movie(sub_raw, sub)

iw_sub = fpl.ImageWidget([sub_raw, sub, sub.std(axis=0)], cmap="viridis", shape=(1, 3), figure_kwargs={"size": (857, 443)}, names=["raw", "%-tile filtered", "std"])

fig_perc_filt = fpl.Figure(size=(991, 301))

fig_perc_filt[0, 0].add_line(sub_raw[:, 0, 0])

perc5 = fig_perc_filt[0, 0].add_line(np.column_stack([np.arange(sub_raw.shape[0]), np.repeat(np.percentile(sub_raw[:, 0, 0], 5), sub_raw.shape[0])]), colors="r", alpha=0.4)

def get_gpcca_path(n_clusters: int = 10):
    return Path("./gpcca_results").joinpath(f"{CURRENT_VID['path'].stem}_rect-{rect_slider.value}_clusters-{n_clusters}.pickle")

GP: pygpcca.GPCCA = pickle.load(open(get_gpcca_path(), "rb"))

n_clusters_slider = IntSlider(value=10, min=10, max=25, description="n_clusters: ")

fig_comps = fpl.Figure(size=(755, 555), names=[["components"]])

LABELS = GP.memberships.reshape(sub.shape[1], sub.shape[2], GP.memberships.shape[-1])

def set_rect(change):
    global BLOCK
    if BLOCK:
        return

    index = change["new"]
    global CURRENT_RECTS
    global CURRENT_RECT
    CURRENT_RECT = CURRENT_RECTS[index]
    

    c0, c1, r0, r1 = CURRENT_RECT

    global movie_raw
    global sub_raw
    global sub
    sub_raw = movie_raw[:, r0:r1:2, c0:c1:2].astype(np.float32).copy()
    sub = np.zeros(sub_raw.shape, dtype=np.float32)
    rolling_percentile_movie(sub_raw, sub)

    std = sub.std(axis=0)
    
    iw_sub.set_data([sub_raw, std])

    for g in fig_perc_filt[0, 0]:
        g.data[:, 1] = 0

    global GP
    GP = pickle.load(open(get_gpcca_path(), "rb"))
    global LABELS
    LABELS = GP.memberships.reshape(sub.shape[1], sub.shape[2], GP.memberships.shape[-1])

    set_clusters({"new": n_clusters_slider.value})

rect_slider.observe(set_rect, "value")

@iw_sub.managed_graphics[0].add_event_handler("click")
def update_percentile_line(ev):
    global sub
    global sub_raw
    global perc5
    
    col, row = ev.pick_info["index"]
    fig_perc_filt[0, 0].graphics[0].data[:, 1] = sub_raw[:, row, col]
    
    perc5.data[:, 1] = sub[:, row, col]
    
    fig_perc_filt[0, 0].auto_scale(maintain_aspect=False)

fig_comps_overlap = fpl.Figure(size=(741, 555), names=[["all components overlaid"]])

def update_labels_percentile(change):
    percentile = change["new"]

    fig_comps_overlap.clear()
    
    for g in fig_comps[0, 0].graphics:
        g.vmin = np.percentile(g.data[:], percentile)
        g.vmax = g.data[:].max()

        alpha = 0.2
        
        rgba_arrays = list()
        rgba = np.zeros([*g.data[:].shape, 4])
        rgba[:, :, :-1] = np.random.rand(3)
        rgba[:, :, -1][g.data[:] < g.vmin] = 0
        rgba[:, :, -1][g.data[:] >= g.vmin] = alpha
        rgba_arrays.append(rgba)
        
        fig_comps_overlap.renderer.blend_mode = "weighted_plus"
        
        for rgba in rgba_arrays:
            fig_comps_overlap[0, 0].add_image(rgba)
            
    fig_comps[0, 0].auto_scale()

comps_percentile_slider = IntSlider(value=80, min=0, max=100, description="percentile")

def set_clusters(change):
    global BLOCK
    if BLOCK:
        return

    global GP
    global LABELS

    img_rows = ceil(sqrt(n_clusters_slider.value))

    fig_comps.clear()

    ix = 0
    for i in range(img_rows):
        for j in range(img_rows):
            if ix > LABELS.shape[-1] - 1:
                break
            img = LABELS.T[ix].T
            fig_comps[0, 0].add_image(img, cmap="viridis", offset=(j * img.shape[1] + 10 * j, i * img.shape[0] + 10 * i, 0))
            ix += 1

    update_labels_percentile({"new": comps_percentile_slider.value})

n_clusters_slider.observe(set_clusters, "value")
set_clusters({"new": 10})

comps_percentile_slider.observe(update_labels_percentile, "value")

for f in [iw_movie.figure, iw_sub.figure, fig_perc_filt, fig_comps, fig_comps_overlap]:
    f.canvas.cursor = "crosshair"

VBox([
    movie_selector,
    rect_slider,
    HBox([iw_movie.show(), VBox([iw_sub.show(), fig_perc_filt.show(maintain_aspect=False)])]),
    n_clusters_slider,
    HBox([fig_comps.show(), fig_comps_overlap.show()]),
    comps_percentile_slider
])